# Open-dLLM Phase 4: Modern Block Diffusion Training

Train a modern block diffusion LLM on multi-source 100B dataset using Kaggle T4 GPU.

- **Model**: ~213M params (16L/1024d/16h/4kv/2816MLP, tied embeddings), seq_len=1024, block_size=32
- **Data**: FinePDFs 50B + DCLM 30B + FineWeb-Edu 20B (pre-shuffled, streaming)
- **Key innovations**: FlexAttention staircase mask, GQA 4:1, Liger fused kernels,
  AMP fp16, gradient checkpointing, CART noise rescheduling, complementary masking,
  WSD scheduler, Muon optimizer, MLP dropout 0.1, torch.compile

In [ ]:
# Install deps — liger-kernel requires CC 7.0+ (T4 = CC 7.5)
!pip install -q torch datasets tokenizers liger-kernel muon

In [ ]:
!git clone https://github.com/hahuyhoang411/Open-dLLM.git
%cd Open-dLLM

In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# Verify T4 (CC 7.5) — required for Liger Kernel + FlexAttention
cc = torch.cuda.get_device_capability()
assert cc >= (7, 0), f"Need CC >= 7.0 for Triton kernels, got {cc}. Switch to T4 accelerator."
print(f"\nCC {cc[0]}.{cc[1]} — Liger Kernel + FlexAttention compatible")

In [ ]:
# Train: 16L/1024d/16h/4kv/2816MLP (~213M params, tied embeddings)
# seq_len=1024, block_size=32, batch=16 (effective: 2x16 x4 accum = 128 samples/step)
# All Phase 4 features: AMP, Liger, FlexAttention, CART, Muon, WSD, MLP dropout 0.1
# VRAM estimate: 213M * 2B (fp16) + optimizer + activations ~ 12-14GB on T4 16GB
!python 04_modern_dllm/modern_dllm.py --train --seq-len 1024 --block-size 32 --batch-size 16 --grad-accum-steps 4

In [ ]:
# Generate sample text
!python 04_modern_dllm/modern_dllm.py --seq-len 1024 --block-size 32 \
    --prompt "The meaning of life is" --max-tokens 128 --denoise-steps 10

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
src = '04_modern_dllm/weights/modern_dllm_b32.pt'
if os.path.exists(src):
    shutil.copy(src, '/kaggle/working/weights/')
    size_mb = os.path.getsize(f'/kaggle/working/weights/{os.path.basename(src)}') / 1e6
    print(f"Saved weights to /kaggle/working/weights/ ({size_mb:.1f} MB)")
else:
    print(f"Weights not found at {src}")